<a href="https://colab.research.google.com/github/gunturuhemanthsaikumar/agentic-rfp-evaluation/blob/main/Agentic_RFP_Evaluation_Cohere_LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RFP Evaluation & Supplier Ranking

**LLM:** Cohere Chat API  \
**Agentic orchestration:** LangGraph  
**Persistence:** SQLite  
**Input:** Four synthetic supplier RFP PDFs

This notebook follows the supplied brief:

1. Load active evaluation criteria from SQLite.
2. Upload and extract supplier PDFs.
3. Use a real Cohere LLM to score every active criterion with **score + justification + evidence**.
4. Validate and normalize the LLM output.
5. Calculate deterministic weighted scores.
6. Calculate criterion benchmarks, gaps, relative performance and weighted PPI.
7. Apply the mandatory deterministic tie-break order.
8. Persist the complete run to SQLite.
9. Display a leaderboard and detailed scorecards.
10. Export the completed run as JSON.

**Important design rule:** the LLM judges proposal content; Python calculates arithmetic, benchmarks, PPI, tie-breaks and final rank. This mirrors the project brief's requirement that the LLM must not decide final arithmetic or ranking.


## What you will need

Before running the notebook:

- A **Cohere API key**.
- The four synthetic supplier RFP PDFs generated for this project:
  - Apex Systems
  - BrightPath Tech
  - NexaWorks
  - Orbit Digital

The source brief requires each artificial proposal to contain an executive summary, solution/implementation approach, timeline/team, price table, security/compliance/risk controls, and support/experience/references, while intentionally creating different strengths, weaknesses, prices, schedules, evidence quality and missing information.

LangGraph is used for the workflow graph.


## 1. Install the required packages

Run this cell once in Google Colab. It installs the PDF parser, Cohere SDK, LangGraph, Pydantic and SQLite/Pandas helpers.


In [2]:
# Google Colab-compatible dependencies
!pip -q install -U cohere langgraph pypdf pydantic "pandas==2.2.3"

print("Dependencies installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 93.7 MB/s eta 0:00:00
Dependencies installed successfully.


## 2. Load the Cohere API key securely

### Recommended Colab method

Use Colab **Secrets**:

1. Open the left sidebar.
2. Click the **key / Secrets** icon.
3. Add a secret named `COHERE_API_KEY`.
4. Paste your Cohere API key.
5. Enable notebook access.

The fallback below also lets you paste the key at runtime if you do not want to use Colab Secrets.

**Never hard-code the API key into the notebook.**


In [3]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    COHERE_API_KEY = userdata.get("COHERE_API_KEY")
except Exception:
    COHERE_API_KEY = None

if not COHERE_API_KEY:
    COHERE_API_KEY = getpass("Enter your Cohere API key: ").strip()

if not COHERE_API_KEY:
    raise ValueError("A Cohere API key is required.")

os.environ["COHERE_API_KEY"] = COHERE_API_KEY
print("Cohere API key loaded successfully.")


Cohere API key loaded successfully.


## 3. Create the Cohere client

We use Cohere's **Chat API V2** and structured JSON output. The model is instructed to return one result for every active criterion.

The current Cohere documentation supports `cohere.ClientV2(...)`, Chat API V2, and JSON/JSON-Schema structured outputs.


In [4]:
import cohere

MODEL_NAME = "command-a-plus-05-2026"
co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])

print("Cohere client created.")
print("Model:", MODEL_NAME)


Cohere client created.
Model: command-a-plus-05-2026


## 4. Define the active evaluation criteria

These are the five criteria specified in the project brief.

| ID | Criterion | Weight | Max score |
|---:|---|---:|---:|
| 1 | Technical Capability | 30% | 10 |
| 2 | Implementation Plan | 20% | 10 |
| 3 | Commercial Value | 20% | 10 |
| 4 | Security & Compliance | 20% | 10 |
| 5 | Support & Experience | 10% | 10 |

The notebook stores these in SQLite so that the evaluation prompt can be built dynamically from the database rather than hard-coded into the LLM prompt.


In [6]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

DB_PATH = Path("rfp_evaluation.db")

criteria_seed = [
    (1, "Technical Capability", "Architecture, integrations, scalability, technical fit", 30.0, 10, 1),
    (2, "Implementation Plan", "Timeline, milestones, staffing, risk plan", 20.0, 10, 1),
    (3, "Commercial Value", "Pricing clarity, total cost, assumptions", 20.0, 10, 1),
    (4, "Security & Compliance", "Controls, certifications, privacy, auditability", 20.0, 10, 1),
    (5, "Support & Experience", "Support model, similar projects, references", 10.0, 10, 1),
]

conn = sqlite3.connect(DB_PATH)
conn.execute("""
CREATE TABLE IF NOT EXISTS evaluation_criteria (
    criterion_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    description TEXT NOT NULL,
    weight REAL NOT NULL,
    max_score REAL NOT NULL,
    is_active INTEGER NOT NULL
)
""")
conn.execute("""
CREATE TABLE IF NOT EXISTS rfp_runs (
    rfp_run_id TEXT PRIMARY KEY,
    created_at TEXT NOT NULL,
    status TEXT NOT NULL
)
""")
conn.execute("""
CREATE TABLE IF NOT EXISTS supplier_results (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    rfp_run_id TEXT NOT NULL,
    supplier_name TEXT NOT NULL,
    submission_date TEXT NOT NULL,
    experience_rating REAL NOT NULL,
    absolute_score REAL NOT NULL,
    ppi REAL NOT NULL,
    final_rank INTEGER NOT NULL,
    result_json TEXT NOT NULL,
    FOREIGN KEY (rfp_run_id) REFERENCES rfp_runs(rfp_run_id)
)
""")

conn.execute("DELETE FROM evaluation_criteria")
conn.executemany(
    "INSERT INTO evaluation_criteria VALUES (?, ?, ?, ?, ?, ?)",
    criteria_seed
)
conn.commit()

criteria_df = pd.read_sql_query(
    "SELECT * FROM evaluation_criteria WHERE is_active = 1 ORDER BY criterion_id",
    conn
)
display(criteria_df)

assert abs(criteria_df["weight"].sum() - 100.0) < 1e-9, "Active weights must total 100%."
print("Active criterion weights total:", criteria_df["weight"].sum(), "%")


,criterion_id,name,description,weight,max_score,is_active
0,1,Technical Capability,"Architecture, integrations, scalability, techn...",30.0,10.0,1
1,2,Implementation Plan,"Timeline, milestones, staffing, risk plan",20.0,10.0,1
2,3,Commercial Value,"Pricing clarity, total cost, assumptions",20.0,10.0,1
3,4,Security & Compliance,"Controls, certifications, privacy, auditability",20.0,10.0,1
4,5,Support & Experience,"Support model, similar projects, references",10.0,10.0,1


Active criterion weights total: 100.0 %


## 5. Upload the four supplier PDFs

Run the next cell and select the four artificial RFP PDFs.

Expected suppliers:

- `Apex Systems`
- `BrightPath Tech`
- `NexaWorks`
- `Orbit Digital`

The notebook does not assume scores. It reads the uploaded proposal text and sends that content to the real Cohere model.


In [7]:
from google.colab import files

uploaded = files.upload()

pdf_paths = sorted([Path(name) for name in uploaded.keys() if name.lower().endswith(".pdf")])

if not pdf_paths:
    raise ValueError("No PDF files were uploaded.")

print("Uploaded PDFs:")
for p in pdf_paths:
    print(" -", p.name)


Saving orbit_digital_artificial_rfp.pdf to orbit_digital_artificial_rfp.pdf
Saving nexaworks_artificial_rfp.pdf to nexaworks_artificial_rfp.pdf
Saving brightpath_tech_artificial_rfp.pdf to brightpath_tech_artificial_rfp.pdf
Saving apex_systems_artificial_rfp.pdf to apex_systems_artificial_rfp.pdf
Uploaded PDFs:
 - apex_systems_artificial_rfp.pdf
 - brightpath_tech_artificial_rfp.pdf
 - nexaworks_artificial_rfp.pdf
 - orbit_digital_artificial_rfp.pdf


## 6. Enter supplier metadata

The brief requires supplier name, submission date and historical experience rating.

For the synthetic documents, use any reasonable fictional metadata. The experience rating is **input metadata**, not an LLM-generated score.

If your four uploaded files use different names, edit the dictionary below.


In [8]:
supplier_metadata = {
    "Apex Systems": {
        "submission_date": "2026-08-20",
        "experience_rating": 8.0
    },
    "BrightPath Tech": {
        "submission_date": "2026-08-22",
        "experience_rating": 5.0
    },
    "NexaWorks": {
        "submission_date": "2026-08-21",
        "experience_rating": 8.5
    },
    "Orbit Digital": {
        "submission_date": "2026-08-19",
        "experience_rating": 9.0
    },
}

print("Supplier metadata loaded:")
display(pd.DataFrame(
    [{"supplier_name": k, **v} for k, v in supplier_metadata.items()]
))


Supplier metadata loaded:


,supplier_name,submission_date,experience_rating
0,Apex Systems,2026-08-20,8.0
1,BrightPath Tech,2026-08-22,5.0
2,NexaWorks,2026-08-21,8.5
3,Orbit Digital,2026-08-19,9.0


## 7. Extract PDF text page-by-page

Each extracted page is labelled as `[Page N]`. This is important because the LLM's evidence field can point back to a specific proposal page.

The extraction step is deterministic Python — no LLM is used here.


In [9]:
from pypdf import PdfReader

def extract_pdf_pages(pdf_path: Path):
    reader = PdfReader(str(pdf_path))
    pages = []
    for idx, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        pages.append({
            "page": idx,
            "text": text.strip()
        })
    return pages

documents = {}

for pdf_path in pdf_paths:
    pages = extract_pdf_pages(pdf_path)
    full_text = "\n\n".join(
        f"[Page {p['page']}]\n{p['text']}" for p in pages
    )
    documents[pdf_path.stem] = {
        "file_name": pdf_path.name,
        "pages": pages,
        "text": full_text
    }

for key, doc in documents.items():
    print(f"{doc['file_name']}: {len(doc['pages'])} pages, {len(doc['text']):,} characters")


apex_systems_artificial_rfp.pdf: 4 pages, 8,345 characters
brightpath_tech_artificial_rfp.pdf: 4 pages, 7,378 characters
nexaworks_artificial_rfp.pdf: 4 pages, 8,236 characters
orbit_digital_artificial_rfp.pdf: 4 pages, 7,658 characters


## 8. Map PDF filenames to supplier names

This helper uses the supplier names expected by the classroom brief. If you renamed the PDFs, update `FILE_TO_SUPPLIER`.



In [10]:
FILE_TO_SUPPLIER = {
    "apex_systems": "Apex Systems",
    "brightpath_tech": "BrightPath Tech",
    "nexaworks": "NexaWorks",
    "orbit_digital": "Orbit Digital",
}

supplier_documents = {}

for stem, doc in documents.items():
    normalized = stem.lower().replace("-", "_").replace(" ", "_")
    supplier_name = FILE_TO_SUPPLIER.get(normalized)

    if supplier_name is None:
        # Try substring matching.
        for token, candidate in FILE_TO_SUPPLIER.items():
            if token in normalized:
                supplier_name = candidate
                break

    if supplier_name is None:
        raise ValueError(
            f"Could not map {doc['file_name']} to a supplier. "
            "Update FILE_TO_SUPPLIER."
        )

    supplier_documents[supplier_name] = doc

missing = set(supplier_metadata) - set(supplier_documents)
if missing:
    raise ValueError(f"Missing uploaded PDFs for: {sorted(missing)}")

print("Supplier/document mapping:")
for supplier, doc in supplier_documents.items():
    print(f" - {supplier} -> {doc['file_name']}")


Supplier/document mapping:
 - Apex Systems -> apex_systems_artificial_rfp.pdf
 - BrightPath Tech -> brightpath_tech_artificial_rfp.pdf
 - NexaWorks -> nexaworks_artificial_rfp.pdf
 - Orbit Digital -> orbit_digital_artificial_rfp.pdf


## 9. Build the LLM evaluation schema

For every active criterion, the Cohere model must return:

- `criterion_id`
- `score` from 0 to 10
- `justification`
- `evidence`
- `evidence_page`

The prompt explicitly says:

1. Use only the supplied proposal.
2. Do not infer missing information.
3. Quote or closely reproduce supporting evidence from the proposal.
4. Give a criterion result for every active criterion.
5. Do not perform final weighted arithmetic or ranking.
6. Return JSON only.

This directly addresses the evidence-grounding requirement in the project brief.


In [14]:
# ============================================================
# STEP — Prepare LLM Evaluation Schema and Prompt
# ============================================================

ACTIVE_CRITERIA = criteria_df.to_dict("records")

evaluation_schema = {
    "type": "object",
    "properties": {
        "supplier_name": {"type": "string"},
        "criteria": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "criterion_id": {"type": "integer"},
                    "score": {"type": "number"},
                    "max_score": {"type": "number"},
                    "justification": {"type": "string"},
                    "evidence": {"type": "string"},
                    "evidence_page": {"type": "integer"},
                },
                "required": ["criterion_id", "score", "max_score", "justification", "evidence", "evidence_page"],
            },
        },
        "risks": {"type": "array", "items": {"type": "string"}},
        "overall_summary": {"type": "string"},
    },
    "required": ["supplier_name", "criteria", "risks", "overall_summary"],
}

def build_evaluation_prompt(supplier_name, proposal_text, criteria):
    criterion_block = "\n".join(
        f"- ID {c['criterion_id']}: {c['name']} | Weight {c['weight']}% | "
        f"Inspect: {c['description']} | Max score {c['max_score']}"
        for c in criteria
    )
    prompt = f"""
You are evaluating ONE supplier proposal for a procurement team.

Your evaluation must be based ONLY on the supplier proposal provided below.

SUPPLIER:
{supplier_name}

ACTIVE EVALUATION CRITERIA:
{criterion_block}

COMPLETE SUPPLIER PROPOSAL:
The proposal below contains the COMPLETE extracted text from the supplier's PDF document.
You must consider information from the ENTIRE proposal, not only the beginning or first page.

--- BEGIN COMPLETE PROPOSAL ---
{proposal_text}
--- END COMPLETE PROPOSAL ---

TASK:
Evaluate the supplier proposal against EVERY active evaluation criterion.

For each criterion:
1. Assign a score from 0 to 10.
2. Explain why that score was assigned.
3. Provide supporting evidence from the proposal.
4. Provide the page number where the evidence appears.
5. Consider both strengths and weaknesses.
6. Consider missing or vague information where relevant.

STRICT EVIDENCE RULES:
- Use ONLY information contained in the supplied proposal.
- You MUST consider the ENTIRE proposal before assigning criterion scores.
- Do NOT assume that a capability exists merely because it would be normal or desirable for a supplier to have.
- Do NOT invent certifications, technologies, integrations, references, experience, prices, timelines, SLAs, security controls, compliance controls, or capabilities.
- If the proposal does not provide sufficient evidence for a criterion, explicitly state: "No explicit evidence provided"
- Missing or vague information should reduce the score when that information is relevant to the criterion.
- Evidence should be a short verbatim excerpt or a highly faithful excerpt from the proposal.
- Do NOT create evidence that is not present in the proposal.
- evidence_page must identify the PDF page containing the supporting evidence.
- If no supporting evidence exists: evidence = "No explicit evidence provided" and evidence_page = 0.
- The score MUST be between 0 and 10 inclusive.
- Return EXACTLY ONE result for EVERY active criterion.
- Use the criterion_id supplied in the active criteria.
- max_score must match the maximum score specified for that criterion.
- Do NOT calculate final weighted score, benchmark, criterion gap, relative performance percentage, Peer Performance Index (PPI), tie-breaks, or final rank.
- Those calculations will be performed separately by deterministic Python business rules.

IMPORTANT:
The purpose of this evaluation is to produce an evidence-grounded assessment of the supplier proposal.
A supplier should receive credit only when the proposal contains supporting evidence.
A supplier should NOT receive additional credit based on general industry expectations or assumptions.

Return JSON only according to the supplied schema.
"""
    return prompt

for supplier_name, document in supplier_documents.items():
    prompt = build_evaluation_prompt(
        supplier_name=supplier_name,
        proposal_text=document["text"],
        criteria=ACTIVE_CRITERIA
    )

    print(f"Evaluation prompt created for: {supplier_name}")
    print(f"Prompt length: {len(prompt):,} characters")

Evaluation prompt created for: Apex Systems
Prompt length: 11,619 characters
Evaluation prompt created for: BrightPath Tech
Prompt length: 10,655 characters
Evaluation prompt created for: NexaWorks
Prompt length: 11,507 characters
Evaluation prompt created for: Orbit Digital
Prompt length: 10,933 characters


## 10. Create the real Cohere evaluation function

This is the first **real LLM** step.

Cohere structured JSON output is used so downstream Python receives a machine-readable object instead of free-form prose.


In [16]:
import json
import time

def cohere_evaluate_supplier(supplier_name, proposal_text):
    prompt = build_evaluation_prompt(
        supplier_name=supplier_name,
        proposal_text=proposal_text,
        criteria=ACTIVE_CRITERIA
    )

    response = co.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a strict procurement proposal evaluator. "
                    "Return only the requested JSON object."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        response_format={
            "type": "json_object",
            "schema": evaluation_schema,
        },
        temperature=0,
        seed=42,
    )

    # Cohere may return multiple content blocks,
    # including thinking and text.
    raw_text = None

    for content_item in response.message.content:
        if content_item.type == "text":
            raw_text = content_item.text
            break

    if raw_text is None:
        raise ValueError(
            "Cohere response did not contain a text output."
        )

    return json.loads(raw_text)


# Test one real LLM call before running the whole batch.
test_supplier = next(iter(supplier_documents))

test_result = cohere_evaluate_supplier(
    test_supplier,
    supplier_documents[test_supplier]["text"]
)

print("Real Cohere evaluation completed")
print(f"Supplier: {test_supplier}")
print()
print(json.dumps(test_result, indent=2))

To suppress this warning, set `log_warning_experimental_features=False` when initializing the client.


Real Cohere evaluation completed
Supplier: Apex Systems

{
  "supplier_name": "Apex Systems",
  "criteria": [
    {
      "criterion_id": 1,
      "score": 7,
      "max_score": 10,
      "justification": "The proposal describes a solid technical architecture with API-first design, containerized services, horizontal scaling, PostgreSQL database, encryption, role-based access, and SSO integration, demonstrating good technical capability. However, it lacks concrete scalability metrics, detailed integration specifications beyond generic adapters, and explicit technology stack details, which limits depth of assessment.",
      "evidence": "Containerized services behind an API gateway. Designed for horizontal scaling; exact peak volumes to be confirmed.",
      "evidence_page": 2
    },
    {
      "criterion_id": 2,
      "score": 8,
      "max_score": 10,
      "justification": "A clear 5-phase implementation timeline with defined durations and milestones is provided, along with a reasona

## 11. Validate and normalize the LLM output

The brief explicitly requires validation and normalization of missing, malformed and out-of-range results.

This function:

- Ensures every active criterion appears exactly once.
- Converts scores to numbers.
- Clips scores to `[0, max_score]`.
- Detects invalid/missing criterion records.
- Detects missing evidence.
- Records warnings instead of silently hiding problems.
- Preserves the LLM's justification and evidence for traceability.

**Important:** this step does not invent a substantive score for a missing criterion. A missing criterion is normalized to `0` with a warning, because the evaluator must not fabricate evidence.


In [17]:
def validate_and_normalize(llm_result, supplier_name, criteria):
    warnings = []
    criterion_by_id = {int(c["criterion_id"]): c for c in criteria}

    if llm_result.get("supplier_name") != supplier_name:
        warnings.append(
            f"Supplier name mismatch: expected '{supplier_name}', "
            f"received '{llm_result.get('supplier_name')}'."
        )

    raw_criteria = llm_result.get("criteria", [])
    normalized = {}

    for item in raw_criteria:
        try:
            cid = int(item.get("criterion_id"))
        except Exception:
            warnings.append("Ignored criterion with invalid criterion_id.")
            continue

        if cid not in criterion_by_id:
            warnings.append(f"Ignored unknown criterion_id={cid}.")
            continue

        if cid in normalized:
            warnings.append(f"Duplicate criterion_id={cid}; first valid result retained.")
            continue

        c = criterion_by_id[cid]
        max_score = float(c["max_score"])

        try:
            score = float(item.get("score"))
        except Exception:
            score = 0.0
            warnings.append(f"Criterion {cid}: malformed score normalized to 0.")

        if score < 0 or score > max_score:
            warnings.append(
                f"Criterion {cid}: score {score} outside [0, {max_score}], clipped."
            )
            score = max(0.0, min(score, max_score))

        justification = str(item.get("justification") or "").strip()
        evidence = str(item.get("evidence") or "").strip()

        if not justification:
            warnings.append(f"Criterion {cid}: missing justification.")
            justification = "No justification returned by the model."

        if not evidence:
            warnings.append(f"Criterion {cid}: missing evidence.")
            evidence = "No explicit evidence provided."

        try:
            evidence_page = int(item.get("evidence_page", 0))
        except Exception:
            evidence_page = 0
            warnings.append(f"Criterion {cid}: invalid evidence_page normalized to 0.")

        normalized[cid] = {
            "criterion_id": cid,
            "score": score,
            "max_score": max_score,
            "justification": justification,
            "evidence": evidence,
            "evidence_page": evidence_page,
        }

    for cid, c in criterion_by_id.items():
        if cid not in normalized:
            warnings.append(
                f"Missing criterion_id={cid} ({c['name']}); normalized to score 0."
            )
            normalized[cid] = {
                "criterion_id": cid,
                "score": 0.0,
                "max_score": float(c["max_score"]),
                "justification": "No criterion result was returned by the model.",
                "evidence": "No explicit evidence provided.",
                "evidence_page": 0,
            }

    normalized_list = [normalized[cid] for cid in sorted(normalized)]

    return {
        "supplier_name": supplier_name,
        "criteria": normalized_list,
        "risks": [str(x) for x in llm_result.get("risks", [])],
        "overall_summary": str(llm_result.get("overall_summary", "")).strip(),
        "warnings": warnings,
    }

validated_test = validate_and_normalize(
    test_result, test_supplier, ACTIVE_CRITERIA
)

print("Validation warnings:")
for warning in validated_test["warnings"]:
    print(" -", warning)

display(pd.DataFrame(validated_test["criteria"]))


Validation warnings:


,criterion_id,score,max_score,justification,evidence,evidence_page
0,1,7.0,10.0,The proposal describes a solid technical archi...,Containerized services behind an API gateway. ...,2
1,2,8.0,10.0,A clear 5-phase implementation timeline with d...,Phase Duration Key milestones Discovery & desi...,2
2,3,7.0,10.0,Pricing is presented in a detailed table with ...,"Estimated total $400,000 $72,000/year",3
3,4,8.0,10.0,"Strong security controls are documented, inclu...",TLS 1.2+ for network communication and AES-256...,3
4,5,6.0,10.0,Support model is well defined with 24x7 incide...,24x7 incident intake for severity-1 production...,4


## 12. Build the LangGraph workflow

This is the agentic orchestration layer.

The graph separates:

1. `load_criteria`
2. `extract_documents`
3. `evaluate_suppliers` — real Cohere LLM
4. `validate_outputs`
5. `calculate_scores`
6. `benchmark_and_rank`
7. `persist_results`

The ranking and arithmetic nodes are deterministic Python. LangGraph controls the order and state flow; the LLM does not decide the final ranking.


In [18]:
from typing import TypedDict, Any
from langgraph.graph import StateGraph, START, END

class RFPState(TypedDict, total=False):
    rfp_run_id: str
    criteria: list
    supplier_documents: dict
    llm_results: dict
    validated_results: dict
    scored_results: dict
    final_results: list
    warnings: list

def node_load_criteria(state: RFPState):
    rows = pd.read_sql_query(
        "SELECT * FROM evaluation_criteria WHERE is_active = 1 ORDER BY criterion_id",
        conn
    )
    if rows.empty:
        raise ValueError("No active evaluation criteria found.")

    if abs(rows["weight"].sum() - 100.0) > 1e-9:
        raise ValueError("Active criterion weights must total 100%.")

    return {"criteria": rows.to_dict("records")}

def node_extract_documents(state: RFPState):
    # Documents were extracted before graph execution so the graph can operate
    # on clean, page-labelled text. This node validates the expected inputs.
    if not supplier_documents:
        raise ValueError("No supplier documents available.")
    return {"supplier_documents": supplier_documents}

def node_evaluate_suppliers(state: RFPState):
    results = {}
    run_warnings = []

    for supplier_name, doc in state["supplier_documents"].items():
        print(f"Calling Cohere for {supplier_name}...")
        raw = cohere_evaluate_supplier(supplier_name, doc["text"])
        results[supplier_name] = raw
        time.sleep(0.5)

    return {"llm_results": results, "warnings": run_warnings}

def node_validate_outputs(state: RFPState):
    validated = {}
    warnings = list(state.get("warnings", []))

    for supplier_name, raw in state["llm_results"].items():
        item = validate_and_normalize(
            raw,
            supplier_name,
            state["criteria"]
        )
        validated[supplier_name] = item
        warnings.extend(
            [f"{supplier_name}: {w}" for w in item["warnings"]]
        )

    return {
        "validated_results": validated,
        "warnings": warnings
    }

def node_calculate_scores(state: RFPState):
    scored = {}

    for supplier_name, result in state["validated_results"].items():
        criterion_rows = []

        absolute_score = 0.0

        for item in result["criteria"]:
            c = next(
                c for c in state["criteria"]
                if int(c["criterion_id"]) == int(item["criterion_id"])
            )

            contribution = (
                float(item["score"]) / float(c["max_score"])
            ) * float(c["weight"])

            absolute_score += contribution

            criterion_rows.append({
                **item,
                "criterion_name": c["name"],
                "weight": float(c["weight"]),
                "weighted_contribution": contribution,
            })

        meta = supplier_metadata[supplier_name]

        scored[supplier_name] = {
            "supplier_name": supplier_name,
            "submission_date": meta["submission_date"],
            "experience_rating": float(meta["experience_rating"]),
            "absolute_score": absolute_score,
            "criteria": criterion_rows,
            "risks": result["risks"],
            "overall_summary": result["overall_summary"],
            "warnings": result["warnings"],
        }

    return {"scored_results": scored}

def node_benchmark_and_rank(state: RFPState):
    scored = state["scored_results"]

    # Benchmark = highest valid score observed for each criterion.
    benchmark_by_id = {}

    for c in state["criteria"]:
        cid = int(c["criterion_id"])
        scores = [
            item["score"]
            for result in scored.values()
            for item in result["criteria"]
            if int(item["criterion_id"]) == cid
        ]
        benchmark_by_id[cid] = max(scores) if scores else 0.0

    # Add benchmark, gap and relative performance.
    for supplier_name, result in scored.items():
        ppi_numerator = 0.0
        ppi_denominator = 0.0

        for item in result["criteria"]:
            cid = int(item["criterion_id"])
            benchmark = float(benchmark_by_id[cid])
            score = float(item["score"])
            weight = float(item["weight"])

            gap = score - benchmark

            if benchmark == 0:
                relative_pct = 100.0 if score == 0 else 0.0
            else:
                relative_pct = (score / benchmark) * 100.0

            item["benchmark"] = benchmark
            item["gap"] = gap
            item["relative_performance_pct"] = relative_pct

            ppi_numerator += relative_pct * weight
            ppi_denominator += weight

        result["ppi"] = (
            ppi_numerator / ppi_denominator
            if ppi_denominator else 0.0
        )

    # Mandatory tie-break order:
    # 1) Higher PPI
    # 2) Earlier submission date
    # 3) Higher historical experience rating
    # 4) Supplier name ascending
    ordered = sorted(
        scored.values(),
        key=lambda x: (
            -float(x["ppi"]),
            x["submission_date"],
            -float(x["experience_rating"]),
            x["supplier_name"].lower()
        )
    )

    for rank, result in enumerate(ordered, start=1):
        result["final_rank"] = rank
        result["tie_break_rule"] = (
            "Higher PPI -> earlier submission date -> higher historical "
            "experience rating -> supplier name ascending"
        )

    return {
        "scored_results": scored,
        "final_results": ordered
    }

def node_persist_results(state: RFPState):
    run_id = state["rfp_run_id"]

    conn.execute(
        "INSERT OR REPLACE INTO rfp_runs VALUES (?, ?, ?)",
        (
            run_id,
            datetime.now(timezone.utc).isoformat(),
            "completed"
        )
    )

    for result in state["final_results"]:
        conn.execute(
            """
            INSERT INTO supplier_results
            (rfp_run_id, supplier_name, submission_date, experience_rating,
             absolute_score, ppi, final_rank, result_json)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                run_id,
                result["supplier_name"],
                result["submission_date"],
                result["experience_rating"],
                result["absolute_score"],
                result["ppi"],
                result["final_rank"],
                json.dumps(result, ensure_ascii=False)
            )
        )

    conn.commit()
    print(f"Persisted run {run_id} to SQLite.")
    return {}

graph_builder = StateGraph(RFPState)

graph_builder.add_node("load_criteria", node_load_criteria)
graph_builder.add_node("extract_documents", node_extract_documents)
graph_builder.add_node("evaluate_suppliers", node_evaluate_suppliers)
graph_builder.add_node("validate_outputs", node_validate_outputs)
graph_builder.add_node("calculate_scores", node_calculate_scores)
graph_builder.add_node("benchmark_and_rank", node_benchmark_and_rank)
graph_builder.add_node("persist_results", node_persist_results)

graph_builder.add_edge(START, "load_criteria")
graph_builder.add_edge("load_criteria", "extract_documents")
graph_builder.add_edge("extract_documents", "evaluate_suppliers")
graph_builder.add_edge("evaluate_suppliers", "validate_outputs")
graph_builder.add_edge("validate_outputs", "calculate_scores")
graph_builder.add_edge("calculate_scores", "benchmark_and_rank")
graph_builder.add_edge("benchmark_and_rank", "persist_results")
graph_builder.add_edge("persist_results", END)

rfp_graph = graph_builder.compile()

print("LangGraph compiled successfully.")


LangGraph compiled successfully.


## 13. Run the complete evaluation

This is the main execution cell.

Because the graph contains a real Cohere call for every supplier, **this cell can incur API usage**.

The resulting state contains:

- raw LLM outputs
- validated scorecards
- deterministic weighted scores
- benchmarks
- gaps
- relative performance
- PPI
- final rank
- warnings


In [19]:
import uuid

RFP_RUN_ID = "RFP_RUN_" + uuid.uuid4().hex[:10].upper()

initial_state = {
    "rfp_run_id": RFP_RUN_ID,
    "supplier_documents": supplier_documents,
    "warnings": [],
}

final_state = rfp_graph.invoke(initial_state)

print("Run completed:", RFP_RUN_ID)
print("Suppliers evaluated:", len(final_state["final_results"]))
print("Warnings:", len(final_state.get("warnings", [])))


To suppress this warning, set `log_warning_experimental_features=False` when initializing the client.


Calling Cohere for Apex Systems...


To suppress this warning, set `log_warning_experimental_features=False` when initializing the client.


Calling Cohere for BrightPath Tech...


To suppress this warning, set `log_warning_experimental_features=False` when initializing the client.


Calling Cohere for NexaWorks...


To suppress this warning, set `log_warning_experimental_features=False` when initializing the client.


Calling Cohere for Orbit Digital...
Persisted run RFP_RUN_E2C3C618F1 to SQLite.
Run completed: RFP_RUN_E2C3C618F1
Suppliers evaluated: 4
Warnings: 0


## 14. Inspect the raw Cohere scorecards

This cell lets you see the actual LLM-generated justification and evidence **before** the deterministic ranking calculations.

This is useful when demonstrating that the application is genuinely evidence-grounded rather than returning placeholder text.


In [20]:
for supplier_name, result in final_state["validated_results"].items():
    print("\n" + "=" * 90)
    print(supplier_name)
    print("=" * 90)
    print("Overall summary:")
    print(result["overall_summary"])

    for item in result["criteria"]:
        print(f"\nCriterion {item['criterion_id']} | Score: {item['score']}/{item['max_score']}")
        print("Justification:", item["justification"])
        print("Evidence:", item["evidence"])
        print("Evidence page:", item["evidence_page"])



Apex Systems
Overall summary:
Apex Systems demonstrates solid technical architecture and implementation planning (scores 7 and 8), with clear commercial pricing but a higher total cost (score 6). Security controls are robust though lacking certifications (score 7), and the support model is defined but references are synthetic (score 5). Key risks include legacy integration complexity, LLM extraction errors, change request impacts, and potential security approval delays.

Criterion 1 | Score: 7.0/10.0
Justification: Assigns 7/10. The proposal details an API‑first architecture with containerized services, horizontal scaling design, PostgreSQL transaction store, and integration adapters for SSO and storage, showing solid technical fit. However, it lacks concrete scalability metrics, peak‑volume confirmation, detailed integration specifications, and performance benchmarks, and legacy integration is only an assumption, limiting the depth of evidence.
Evidence: Containerized services behind

## 15. Calculate and display the absolute weighted score

The required formula is:

**Absolute weighted score = Σ ((criterion score / maximum score) × criterion weight)**

The LLM does **not** calculate this number. Python does it after validation.


In [21]:
leaderboard = pd.DataFrame([
    {
        "Rank": r["final_rank"],
        "Supplier": r["supplier_name"],
        "Absolute Score": round(r["absolute_score"], 2),
        "PPI": round(r["ppi"], 2),
        "Submission Date": r["submission_date"],
        "Experience Rating": r["experience_rating"],
    }
    for r in final_state["final_results"]
]).sort_values("Rank")

display(leaderboard)


,Rank,Supplier,Absolute Score,PPI,Submission Date,Experience Rating
0,1,NexaWorks,72.0,94.92,2026-08-21,8.5
1,2,Apex Systems,68.0,90.56,2026-08-20,8.0
2,3,Orbit Digital,69.0,90.00,2026-08-19,9.0
3,4,BrightPath Tech,61.0,80.20,2026-08-22,5.0


## 16. Inspect criterion benchmarks, gaps and relative performance

For each criterion:

- **Benchmark** = highest valid supplier score for that criterion.
- **Gap** = supplier score − benchmark.
- **Relative performance %** = supplier score / benchmark × 100.
- If the benchmark is zero, the notebook uses a safe rule to avoid division by zero.

The weighted average of relative-performance percentages becomes the **Peer Performance Index (PPI)**.


In [22]:
benchmark_rows = []

for result in final_state["final_results"]:
    for item in result["criteria"]:
        benchmark_rows.append({
            "Supplier": result["supplier_name"],
            "Criterion": item["criterion_name"],
            "Weight %": item["weight"],
            "Score": item["score"],
            "Benchmark": item["benchmark"],
            "Gap": item["gap"],
            "Relative %": round(item["relative_performance_pct"], 2),
            "Weighted Contribution": round(item["weighted_contribution"], 2),
        })

benchmark_df = pd.DataFrame(benchmark_rows)
display(benchmark_df)


,Supplier,Criterion,Weight %,Score,Benchmark,Gap,Relative %,Weighted Contribution
0,NexaWorks,Technical Capability,30.0,7.0,7.0,0.0,100.00,21.0
1,NexaWorks,Implementation Plan,20.0,8.0,8.0,0.0,100.00,16.0
2,NexaWorks,Commercial Value,20.0,8.0,8.0,0.0,100.00,16.0
3,NexaWorks,Security & Compliance,20.0,6.0,7.0,-1.0,85.71,12.0
4,NexaWorks,Support & Experience,10.0,7.0,9.0,-2.0,77.78,7.0
5,Apex Systems,Technical Capability,30.0,7.0,7.0,0.0,100.00,21.0
6,Apex Systems,Implementation Plan,20.0,8.0,8.0,0.0,100.00,16.0
7,Apex Systems,Commercial Value,20.0,6.0,8.0,-2.0,75.00,12.0
8,Apex Systems,Security & Compliance,20.0,7.0,7.0,0.0,100.00,14.0
9,Apex Systems,Support & Experience,10.0,5.0,9.0,-4.0,55.56,5.0


## 17. Inspect detailed supplier scorecards

This is the notebook equivalent of the detailed scorecard screen required by the brief, without using Streamlit.

For every criterion you can see:

- score
- weight
- weighted contribution
- benchmark
- gap
- relative performance
- justification
- evidence
- source page


In [23]:
def show_scorecard(supplier_name):
    result = next(
        r for r in final_state["final_results"]
        if r["supplier_name"] == supplier_name
    )

    print(f"\n{'='*100}")
    print(f"{supplier_name} — Detailed Scorecard")
    print(f"{'='*100}")
    print(f"Absolute score: {result['absolute_score']:.2f}")
    print(f"PPI: {result['ppi']:.2f}")
    print(f"Final rank: {result['final_rank']}")
    print(f"Submission date: {result['submission_date']}")
    print(f"Historical experience rating: {result['experience_rating']}")

    for item in result["criteria"]:
        print(f"\n--- {item['criterion_name']} ---")
        print(f"Score: {item['score']}/{item['max_score']}")
        print(f"Weight: {item['weight']}%")
        print(f"Weighted contribution: {item['weighted_contribution']:.2f}")
        print(f"Benchmark: {item['benchmark']}")
        print(f"Gap: {item['gap']}")
        print(f"Relative performance: {item['relative_performance_pct']:.2f}%")
        print(f"Justification: {item['justification']}")
        print(f"Evidence: {item['evidence']}")
        print(f"Evidence page: {item['evidence_page']}")

show_scorecard("NexaWorks")



NexaWorks — Detailed Scorecard
Absolute score: 72.00
PPI: 94.92
Final rank: 1
Submission date: 2026-08-21
Historical experience rating: 8.5

--- Technical Capability ---
Score: 7.0/10.0
Weight: 30.0%
Weighted contribution: 21.00
Benchmark: 7.0
Gap: 0.0
Relative performance: 100.00%
Justification: The proposal provides a clear, modular architecture with separation of LLM evaluation and deterministic ranking, and defines data storage and versioning. Integration scope is defined (SSO, email, REST APIs) with a discovery approach for additional integrations, and an integration allowance is budgeted. However, scalability details (e.g., capacity beyond 1,500 docs/month) and technical specifics (cloud platform, multi-tenancy) are not explicitly described, which limits full assessment.
Evidence: “Modular services with clear separation between LLM evaluation and deterministic ranking. Business rules remain outside the LLM. Integration: SSO, email and REST APIs in base scope; other integrations 

## 18. Explain the deterministic ranking and tie-breaks

The required tie-break order is:

1. Higher PPI first
2. Earlier submission date
3. Higher historical experience rating
4. Supplier name in ascending order

Only after this stable sort are ranks assigned as `1, 2, 3, ...`.


In [24]:
print("Final deterministic ordering:")
for r in final_state["final_results"]:
    print(
        f"Rank {r['final_rank']}: {r['supplier_name']} | "
        f"PPI={r['ppi']:.2f} | "
        f"Submission={r['submission_date']} | "
        f"Experience={r['experience_rating']}"
    )

print("\nTie-break rule:")
print(final_state["final_results"][0]["tie_break_rule"])


Final deterministic ordering:
Rank 1: NexaWorks | PPI=94.92 | Submission=2026-08-21 | Experience=8.5
Rank 2: Apex Systems | PPI=90.56 | Submission=2026-08-20 | Experience=8.0
Rank 3: Orbit Digital | PPI=90.00 | Submission=2026-08-19 | Experience=9.0
Rank 4: BrightPath Tech | PPI=80.20 | Submission=2026-08-22 | Experience=5.0

Tie-break rule:
Higher PPI -> earlier submission date -> higher historical experience rating -> supplier name ascending


## 19. Inspect validation warnings

Warnings are intentionally visible.

Examples include:

- missing criterion
- invalid score
- score outside 0–10
- duplicate criterion
- missing justification
- missing evidence
- invalid evidence page

This makes the pipeline auditable instead of silently repairing LLM output.


In [25]:
warnings = final_state.get("warnings", [])

if warnings:
    print("Validation warnings:")
    for w in warnings:
        print(" -", w)
else:
    print("No validation warnings were recorded.")


No validation warnings were recorded.


## 20. Verify SQLite persistence

The project brief requires the complete run to be stored using one `RFP_RUN_ID`.

The database contains:

- `evaluation_criteria`
- `rfp_runs`
- `supplier_results`


In [26]:
stored_runs = pd.read_sql_query(
    "SELECT * FROM rfp_runs ORDER BY created_at DESC",
    conn
)
stored_results = pd.read_sql_query(
    """
    SELECT rfp_run_id, supplier_name, submission_date,
           experience_rating, absolute_score, ppi, final_rank
    FROM supplier_results
    WHERE rfp_run_id = ?
    ORDER BY final_rank
    """,
    conn,
    params=(RFP_RUN_ID,)
)

print("Stored run:")
display(stored_runs[stored_runs["rfp_run_id"] == RFP_RUN_ID])

print("Stored supplier results:")
display(stored_results)


Stored run:


,rfp_run_id,created_at,status
0,RFP_RUN_E2C3C618F1,2026-08-29T18:10:44.598379+00:00,completed


Stored supplier results:


,rfp_run_id,supplier_name,submission_date,experience_rating,absolute_score,ppi,final_rank
0,RFP_RUN_E2C3C618F1,NexaWorks,2026-08-21,8.5,72.0,94.920635,1
1,RFP_RUN_E2C3C618F1,Apex Systems,2026-08-20,8.0,68.0,90.555556,2
2,RFP_RUN_E2C3C618F1,Orbit Digital,2026-08-19,9.0,69.0,90.000000,3
3,RFP_RUN_E2C3C618F1,BrightPath Tech,2026-08-22,5.0,61.0,80.198413,4


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## 21. Export the complete result as JSON

The final JSON contains the validated evidence, criterion calculations, benchmarks, PPI, ranking metadata and warnings.


In [27]:
export_payload = {
    "rfp_run_id": RFP_RUN_ID,
    "criteria": final_state["criteria"],
    "suppliers": final_state["final_results"],
    "run_warnings": final_state.get("warnings", []),
}

JSON_OUTPUT = Path(f"{RFP_RUN_ID}_complete_result.json")
JSON_OUTPUT.write_text(
    json.dumps(export_payload, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"JSON exported to: {JSON_OUTPUT}")
print(f"File size: {JSON_OUTPUT.stat().st_size:,} bytes")


JSON exported to: RFP_RUN_E2C3C618F1_complete_result.json
File size: 28,035 bytes


## 22. Demonstration: validation/error case

To demonstrate the required validation/error case, this cell intentionally creates a malformed LLM-like response **without sending it to Cohere**.

This is not a mock of the production evaluation. It is only a unit-style demonstration of the validation tool required by the brief.

The production path remains real Cohere → validation → deterministic scoring.


In [28]:
malformed_example = {
    "supplier_name": "Example Supplier",
    "criteria": [
        {
            "criterion_id": 1,
            "score": 14,  # invalid: above max 10
            "max_score": 10,
            "justification": "",
            "evidence": "",
            "evidence_page": "not-a-number"
        },
        # Criterion 2 is deliberately missing.
    ],
    "risks": [],
    "overall_summary": ""
}

validated_error_case = validate_and_normalize(
    malformed_example,
    "Example Supplier",
    ACTIVE_CRITERIA
)

print("Validation warnings:")
for warning in validated_error_case["warnings"]:
    print(" -", warning)

print("\nNormalized criteria:")
display(pd.DataFrame(validated_error_case["criteria"]))


Validation warnings:
 - Criterion 1: score 14.0 outside [0, 10.0], clipped.
 - Criterion 1: missing justification.
 - Criterion 1: missing evidence.
 - Criterion 1: invalid evidence_page normalized to 0.
 - Missing criterion_id=2 (Implementation Plan); normalized to score 0.
 - Missing criterion_id=3 (Commercial Value); normalized to score 0.
 - Missing criterion_id=4 (Security & Compliance); normalized to score 0.
 - Missing criterion_id=5 (Support & Experience); normalized to score 0.

Normalized criteria:


,criterion_id,score,max_score,justification,evidence,evidence_page
0,1,10.0,10.0,No justification returned by the model.,No explicit evidence provided.,0
1,2,0.0,10.0,No criterion result was returned by the model.,No explicit evidence provided.,0
2,3,0.0,10.0,No criterion result was returned by the model.,No explicit evidence provided.,0
3,4,0.0,10.0,No criterion result was returned by the model.,No explicit evidence provided.,0
4,5,0.0,10.0,No criterion result was returned by the model.,No explicit evidence provided.,0
